# Mode 2 — Probe 2 v2 (binary, Sys3 sweep)

Focused notebook: binary Probe 2 trained on `confirmed_keep` vs `synthetic_spurious_parallel`, evaluated with Sys3 (probe alone) across all four training prompt variants. Reports both macro F1 (all plans) and enrichable F1 (plans with ≥1 GT edge to remove).

**Files needed in `/content/`:**

| File | Purpose |
|---|---|
| `proScript_data-20260226T065943Z-1-001.zip` | ProScript JSON files |
| `proscript_train_edges_v2.csv` | `label=1` confirmed edges → `confirmed_keep` |
| `proscript_train_edges_v3.csv` | Identifies multi-ordering plans |
| `proscript_pipeline_eval_final.csv` | ProScript eval (`ground_truth_removed_edges`) |
| `captaincook_pipeline_eval.csv` | CaptainCook zero-shot transfer |


In [ ]:
import subprocess, zipfile, os
subprocess.run(['pip','install','-q','transformers','accelerate','scikit-learn','tqdm'], check=True)
BASE_DIR = '/content'
DATA_DIR = '/content/proScript_data'
os.makedirs(DATA_DIR, exist_ok=True)
zip_c = [f for f in os.listdir(BASE_DIR) if 'proScript_data' in f and f.endswith('.zip')]
if len([f for f in os.listdir(DATA_DIR) if f.endswith('.json')]) < 600:
    assert zip_c, 'Upload proScript_data*.zip'
    with zipfile.ZipFile(os.path.join(BASE_DIR, zip_c[0])) as z: z.extractall(BASE_DIR)
n_json = len([f for f in os.listdir(DATA_DIR) if f.endswith('.json')])
assert n_json >= 600, f'Expected ~622 JSON files, found {n_json}'
required = ['proscript_train_edges_v2.csv','proscript_train_edges_v3.csv',
            'proscript_pipeline_eval_final.csv','captaincook_pipeline_eval.csv']
missing = [f for f in required if not os.path.exists(os.path.join(BASE_DIR, f))]
assert not missing, f'Missing: {missing}'
print(f'✓ {n_json} JSON files  ✓ all CSVs present')


✓ 622 JSON files  ✓ all CSVs present


In [ ]:
import json as _json, random, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')
CONTENT = Path('/content')

cfg = {
    'model_name':     'mistralai/Mistral-7B-Instruct-v0.1',
    'probe_layer':    17,
    'retrain_probes': True,
    # Threshold sweep values for p2_rem_t
    'sweep_thresholds': [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    'max_iterations': 10,
    'seed': 42,
    'chat_template': True,
}
SEED = cfg['seed']
print('Config:')
print(_json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))


Config:
{
  "model_name": "mistralai/Mistral-7B-Instruct-v0.1",
  "probe_layer": "17",
  "retrain_probes": "True",
  "sweep_thresholds": "[0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]",
  "max_iterations": "10",
  "seed": "42",
  "chat_template": "True"
}


In [ ]:
import json as _json, os, re
from collections import defaultdict, deque

def parse_plan_json(goal, data_dir):
    path = os.path.join(data_dir, goal.replace(' ','_') + '.json')
    if not os.path.exists(path): return None
    with open(path, encoding='utf-8-sig') as f: raw = f.read().strip()
    if not raw: return None
    d = _json.loads(raw)
    steps = {str(k): v for k, v in d['steps'].items()}
    edges_raw = [(str(a), str(b)) for a, b in d['edges']]
    step_map, start_node, end_node = {}, None, None
    for k, v in steps.items():
        ki, vs = int(k), v.strip()
        if vs.upper() == 'START': start_node = ki
        elif vs.upper() == 'END':  end_node = ki
        else:                       step_map[ki] = vs
    real = list(step_map.keys())
    adj = defaultdict(list)
    dag_edges = []
    for a_s, b_s in edges_raw:
        a, b = int(a_s), int(b_s)
        if start_node is not None and a == start_node: continue
        if end_node   is not None and b == end_node:   continue
        if a in step_map and b in step_map:
            adj[a].append(b)
            dag_edges.append((step_map[a], step_map[b]))
    def reach(s):
        v, q = set(), [s]
        while q:
            n = q.pop()
            for nb in adj.get(n, []):
                if nb not in v: v.add(nb); q.append(nb)
        return v
    r = {n: reach(n) for n in real}
    incompat = [(step_map[n1], step_map[n2])
                for i, n1 in enumerate(real)
                for n2 in real[i+1:]
                if n2 not in r[n1] and n1 not in r[n2]]
    return {'steps': step_map, 'dag_edges': dag_edges, 'incomparable': incompat}

def compute_step_depths(steps_list, edges_list):
    """Longest-path depth from any source to each step."""
    adj = defaultdict(set)
    in_deg = {s: 0 for s in steps_list}
    for a, b in edges_list:
        adj[a].add(b)
        in_deg[b] = in_deg.get(b, 0) + 1
    depth = {s: 0 for s in steps_list}
    q = deque([s for s in steps_list if in_deg.get(s, 0) == 0])
    while q:
        n = q.popleft()
        for nb in adj.get(n, set()):
            depth[nb] = max(depth[nb], depth[n] + 1)
            in_deg[nb] -= 1
            if in_deg[nb] == 0: q.append(nb)
    return depth

def parse_edges(s):
    if not s or str(s) == 'nan': return []
    return [(a.strip(), b.strip()) for e in str(s).split(' | ')
            if '→' in e for a, b in [e.split('→', 1)]]

def parse_steps(row):
    return [s.strip() for s in str(row.get('steps', '')).split('|') if s.strip()]

def parse_gt(s): return set(map(tuple, parse_edges(s)))

def build_adj(edges):
    adj = defaultdict(set)
    for a, b in edges: adj[a].add(b)
    return adj

def build_all_pairs_from_edges(steps_list, edges_list):
    """All directed incomparable pairs given steps + edge lists."""
    adj = build_adj(edges_list)
    def get_r(s):
        v, q = set(), [s]
        while q:
            n = q.pop()
            for nb in adj.get(n, set()):
                if nb not in v: v.add(nb); q.append(nb)
        return v
    r = {s: get_r(s) for s in steps_list}
    return {(a, b) for a in steps_list for b in steps_list
            if a != b and b not in r[a]}

print('Utils ready')


Utils ready


In [ ]:
def wrap(raw):
    return f'[INST] {raw.strip()} [/INST]' if cfg['chat_template'] else raw

# ── Four training prompt variants ────────────────────────────────────────────
# Each encodes a different framing of the ordering question.
# The hidden state at the last token is what trains and scores Probe 2.
# p_yes is only meaningful for tp1 and tp4 (yes/no answer format).

def tp1_ordering(goal, a, b):
    return wrap(
        f'You are judging a temporal dependency between two actions in a task.\n'
        f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
        f'Question: Must Action A happen before Action B? Answer yes or no.\n'
        f'Answer:')

def tp2_parallel(goal, a, b):
    return wrap(
        f'You are analyzing whether two actions in a task can be done in parallel.\n'
        f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
        f'Question: Can Action A and Action B be performed at the same time, '
        f'or does one need to happen first? Answer "parallel" or "sequential".\n'
        f'Answer:')

def tp3_dependency(goal, a, b):
    return wrap(
        f'You are checking whether one action must finish before another can start.\n'
        f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
        f'Question: Does Action A need to finish before Action B can begin, '
        f'or can they be done in either order? '
        f'Answer "must finish first" or "either order".\n'
        f'Answer:')

def tp4_flexibility(goal, a, b):
    return wrap(
        f'You are deciding whether an ordering constraint between two steps is necessary.\n'
        f'Task: {goal}\nStep A: {a}\nStep B: {b}\n'
        f'Question: If you swap the order — doing B before A — '
        f'would the task still complete correctly? Answer yes or no.\n'
        f'Answer:')

TRAIN_PROMPT_VARIANTS = {
    'tp1_ordering':    tp1_ordering,
    'tp2_parallel':    tp2_parallel,
    'tp3_dependency':  tp3_dependency,
    'tp4_flexibility': tp4_flexibility,
}
print('Training prompt variants:', list(TRAIN_PROMPT_VARIANTS))


Training prompt variants: ['tp1_ordering', 'tp2_parallel', 'tp3_dependency', 'tp4_flexibility']


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
print(f'Loading {cfg["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
model = AutoModelForCausalLM.from_pretrained(
    cfg['model_name'], torch_dtype=torch.float16,
    device_map='auto', output_hidden_states=True)
model.eval()
YES_IDS = [tokenizer.encode(t, add_special_tokens=False)[0]
           for t in (' yes', 'yes', 'Yes')]
NO_IDS  = [tokenizer.encode(t, add_special_tokens=False)[0]
           for t in (' no',  'no',  'No')]
print(f'Model loaded  |  YES: {YES_IDS}  |  NO: {NO_IDS}')


Loading mistralai/Mistral-7B-Instruct-v0.1 ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded  |  YES: [5081, 5081, 5592]  |  NO: [708, 708, 1770]


In [ ]:
@torch.no_grad()
def get_hidden_and_pyes(prompt, layer):
    """
    Single forward pass → (hidden_vec, p_yes_normalised).
    p_yes is the normalised probability of yes/(yes+no) at the last token.
    Used by both Probe 2 (hidden state) and LLM baselines (p_yes).
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out    = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[layer][0, -1].float().cpu().numpy()
    probs  = torch.softmax(out.logits[0, -1].float(), dim=-1).cpu()
    p_yes  = sum(probs[i].item() for i in YES_IDS)
    p_no   = sum(probs[i].item() for i in NO_IDS)
    denom  = p_yes + p_no if (p_yes + p_no) > 0 else 1.0
    return hidden, p_yes / denom

print('Feature helper ready')


Feature helper ready


In [ ]:
# ── Load CSVs ──────────────────────────────────────────────────────────────
base_v2  = pd.read_csv(CONTENT / 'proscript_train_edges_v2.csv')
base_v3  = pd.read_csv(CONTENT / 'proscript_train_edges_v3.csv')
ps_eval  = pd.read_csv(CONTENT / 'proscript_pipeline_eval_final.csv')
cc_eval  = pd.read_csv(CONTENT / 'captaincook_pipeline_eval.csv')

eval_goals  = set(ps_eval['goal']) | set(cc_eval['goal'])
multi_goals = set(base_v3[base_v3['edge_type'] == 'truly_parallel']['goal']) - eval_goals

# ── Probe 2 training classes ──────────────────────────────────────────────────
#
# Class 1 — confirmed_keep:
#   Existing DAG edges that are real ordering constraints (label=1 in v2).
#   At inference: Probe 2 sees an existing edge and should score it as 'keep'.
#
# Class 0 — synthetic_spurious_parallel:
#   Same-depth incomparable pairs from multi-ordering training plans,
#   presented as candidate edges. These represent steps that could run in
#   parallel — no genuine ordering dependency between them.
#   At inference: the probe generalises from 'no dependency' hidden states
#   to 'spurious/over-constrained edge' hidden states.

confirmed_keep = base_v2[
    (base_v2['label'] == 1) & (~base_v2['goal'].isin(eval_goals))
].copy()

synthetic_rows = []
for goal in sorted(multi_goals):
    plan = parse_plan_json(goal, DATA_DIR)
    if not plan: continue
    steps_list = list(plan['steps'].values())
    depths     = compute_step_depths(steps_list, plan['dag_edges'])
    for a, b in plan['incomparable']:
        if depths.get(a) == depths.get(b):    # same depth only
            synthetic_rows.append({'goal': goal, 'a': a, 'b': b, 'probe_label': 0})
            synthetic_rows.append({'goal': goal, 'a': b, 'b': a, 'probe_label': 0})
synthetic_df = pd.DataFrame(synthetic_rows)

n_each = min(len(confirmed_keep), len(synthetic_df))
rng    = np.random.default_rng(SEED)
probe2_train = pd.concat([
    confirmed_keep.sample(n_each, random_state=SEED).assign(probe_label=1),
    synthetic_df.sample(n_each,   random_state=SEED).assign(probe_label=0),
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'confirmed_keep:          {len(confirmed_keep):>5} rows')
print(f'synthetic_spurious:      {len(synthetic_df):>5} rows')
print(f'probe2_train (balanced): {len(probe2_train):>5} rows  '
      f'({n_each} per class)  plans={probe2_train.goal.nunique()}')

# Eval leakage check
assert len(set(probe2_train.goal) & eval_goals) == 0, 'Eval leakage in training data!'
print('✓ No eval leakage')
print(f'\nProScript eval: {len(ps_eval)} plans  '
      f'(enrichable: {(ps_eval["ground_truth_removed_edges"].notna() & (ps_eval["ground_truth_removed_edges"] != "")).sum()})')
print(f'CaptainCook eval: {len(cc_eval)} plans')


confirmed_keep:            718 rows
synthetic_spurious:        502 rows
probe2_train (balanced):  1004 rows  (502 per class)  plans=363
✓ No eval leakage

ProScript eval: 91 plans  (enrichable: 39)
CaptainCook eval: 21 plans


In [ ]:
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# probes2[tp_name] = binary Probe 2 for each training prompt variant
# Classes: 0 = synthetic_spurious (remove), 1 = confirmed_keep (keep)
probes2    = {}
probe2_cv  = {}   # tp_name -> list of fold AUCs
gkf        = GroupKFold(n_splits=5)
plan_groups = probe2_train['goal'].values

if not cfg['retrain_probes']:
    for tp in TRAIN_PROMPT_VARIANTS:
        p_path = CONTENT / f'probe2_{tp}_layer{cfg["probe_layer"]}.pkl'
        assert p_path.exists(), f'Missing {p_path}; set retrain_probes=True'
        with open(p_path, 'rb') as f: probes2[tp] = pickle.load(f)
    print('Loaded saved probes:', list(probes2))
else:
    for tp_name, tp_fn in TRAIN_PROMPT_VARIANTS.items():
        print(f'\n=== Extracting hidden states: {tp_name} ===')
        X, y = [], []
        for _, row in tqdm(probe2_train.iterrows(), total=len(probe2_train)):
            # Corrected: Call get_hidden_and_pyes and get the first return value (hidden state)
            h, _ = get_hidden_and_pyes(tp_fn(row['goal'], row['a'], row['b']), cfg['probe_layer'])
            X.append(h)
            y.append(row['probe_label'])
        X, y = np.array(X), np.array(y)

        # 5-fold plan-level CV
        fold_aucs = []
        for tr, val in gkf.split(X, y, groups=plan_groups):
            p = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
            p.fit(X[tr], y[tr])
            fold_aucs.append(roc_auc_score(y[val], p.predict_proba(X[val])[:, 1]))
        probe2_cv[tp_name] = fold_aucs
        print(f'  CV AUC {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}  '
              f'folds: {[round(a, 4) for a in fold_aucs]}')

        # Final probe on full training set
        probe_f = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
        probe_f.fit(X, y)
        probes2[tp_name] = probe_f

        save_path = CONTENT / f'probe2_{tp_name}_layer{cfg["probe_layer"]}.pkl'
        with open(save_path, 'wb') as f: pickle.dump(probe_f, f)
        print(f'  Saved → {save_path.name}')

    print('\n=== Probe 2 CV summary ===')
    for tp, aucs in probe2_cv.items():
        print(f'  {tp:22}: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

def probe2_p_spurious(h, tp_name):
    """
    Returns P(spurious/remove) for a hidden state.
    Binary probe: class 0 = synthetic_spurious, class 1 = confirmed_keep.
    sklearn orders classes by label value, so index 0 = P(label=0) = P(spurious).
    """
    return probes2[tp_name].predict_proba(h.reshape(1, -1))[0, 0]

print('\nAll Probe 2 variants ready:', list(probes2))


=== Extracting hidden states: tp1_ordering ===


  0%|          | 0/1004 [00:00<?, ?it/s]

  CV AUC 0.9020 ± 0.0263  folds: [np.float64(0.9477), np.float64(0.8837), np.float64(0.8822), np.float64(0.916), np.float64(0.8805)]
  Saved → probe2_tp1_ordering_layer17.pkl

=== Extracting hidden states: tp2_parallel ===


  0%|          | 0/1004 [00:00<?, ?it/s]

  CV AUC 0.9052 ± 0.0181  folds: [np.float64(0.9392), np.float64(0.8871), np.float64(0.8974), np.float64(0.9063), np.float64(0.8959)]
  Saved → probe2_tp2_parallel_layer17.pkl

=== Extracting hidden states: tp3_dependency ===


  0%|          | 0/1004 [00:00<?, ?it/s]

  CV AUC 0.8873 ± 0.0256  folds: [np.float64(0.9298), np.float64(0.8745), np.float64(0.8566), np.float64(0.9013), np.float64(0.8744)]
  Saved → probe2_tp3_dependency_layer17.pkl

=== Extracting hidden states: tp4_flexibility ===


  0%|          | 0/1004 [00:00<?, ?it/s]

  CV AUC 0.9044 ± 0.0223  folds: [np.float64(0.9376), np.float64(0.8726), np.float64(0.8996), np.float64(0.919), np.float64(0.8932)]
  Saved → probe2_tp4_flexibility_layer17.pkl

=== Probe 2 CV summary ===
  tp1_ordering          : 0.9020 ± 0.0263
  tp2_parallel          : 0.9052 ± 0.0181
  tp3_dependency        : 0.8873 ± 0.0256
  tp4_flexibility       : 0.9044 ± 0.0223

All Probe 2 variants ready: ['tp1_ordering', 'tp2_parallel', 'tp3_dependency', 'tp4_flexibility']


In [ ]:
def mode2_metrics(proposed_removals, gt_removed, all_existing_edges):
    """
    Evaluate proposed edge removals against ground truth.
    Restricts GT to edges present in all_existing_edges (reachable_gt).
    """
    reachable_gt = gt_removed & set(all_existing_edges)
    tp = len(proposed_removals & reachable_gt)
    fp = len(proposed_removals - reachable_gt)
    fn = len(reachable_gt - proposed_removals)
    n_neg   = len(all_existing_edges) - len(reachable_gt)
    fp_rate = fp / n_neg if n_neg > 0 else 0.0
    prec    = tp / len(proposed_removals) if proposed_removals else 0.0
    rec     = tp / len(reachable_gt)      if reachable_gt      else 0.0
    f1      = 2*prec*rec/(prec+rec)       if (prec+rec)        else 0.0
    return dict(precision=prec, recall=rec, f1=f1, fp_rate=fp_rate,
                tp=tp, fp=fp, fn=fn,
                n_proposed=len(proposed_removals),
                n_reachable_gt=len(reachable_gt))

def summarise(rows, label=''):
    """
    Two summary metrics:
      macro_f1:      mean per-plan F1 over ALL plans (includes 0 for non-enrichable)
      enrichable_f1: mean per-plan F1 restricted to plans with ≥1 GT edge to remove
      corpus_f1:     micro F1 computed from total TP/FP/FN across all plans
    """
    df = pd.DataFrame(rows)
    macro_f1 = df['f1'].mean()

    enrichable = df[df['n_reachable_gt'] > 0]
    enrichable_f1 = enrichable['f1'].mean() if len(enrichable) > 0 else 0.0

    total_tp = df['tp'].sum(); total_fp = df['fp'].sum(); total_fn = df['fn'].sum()
    c_prec = total_tp/(total_tp+total_fp) if (total_tp+total_fp) else 0.0
    c_rec  = total_tp/(total_tp+total_fn) if (total_tp+total_fn) else 0.0
    corpus_f1 = 2*c_prec*c_rec/(c_prec+c_rec) if (c_prec+c_rec) else 0.0

    result = dict(
        macro_f1       = round(macro_f1, 4),
        enrichable_f1  = round(enrichable_f1, 4),
        corpus_f1      = round(corpus_f1, 4),
        corpus_prec    = round(c_prec, 4),
        corpus_rec     = round(c_rec, 4),
        fp_rate        = round(df['fp_rate'].mean(), 4),
        n_enrichable   = len(enrichable),
        n_total_plans  = len(df),
        avg_iterations = round(df['n_iterations'].mean(), 2) if 'n_iterations' in df else None,
    )
    if label:
        print(f'  {label}: macro_f1={result["macro_f1"]}  '
              f'enrichable_f1={result["enrichable_f1"]}  '
              f'corpus_f1={result["corpus_f1"]}  '
              f'prec={result["corpus_prec"]}  rec={result["corpus_rec"]}  '
              f'fp_rate={result["fp_rate"]}')
    return result

print('Eval helpers ready')


Eval helpers ready


---
## Mode 2 — Probe 2 (binary) + LLM baselines

**Probe 2 (binary):** same iterative loop, threshold sweep over `p_spurious`.

**LLM baselines** both use `tp1_ordering` p_yes, cached during the probe evaluation pass:

- **LLM direct** — `p_yes < t` → propose removal. Assumes low ordering confidence = spurious edge.
- **LLM inverted** — `p_yes > t` → propose removal. Exploits the yes-bias: the model scores spurious edges *higher* on p_yes than confirmed edges (Corr ≈ −0.38), so high p_yes is the better removal signal.

Both baselines use the same iterative loop and threshold sweep. p_yes is extracted once per edge during the `tp1_ordering` probe pass — no additional inference needed.


In [ ]:
def apply_removals(current_edges, removals):
    rem_set = set(removals)
    removed = [e for e in current_edges if e in rem_set]
    edges   = [e for e in current_edges if e not in rem_set]
    return edges, removed

def run_iterative_mode2(goal, steps_list, init_edges, tp_name, tp_fn, threshold):
    """
    Iteratively remove edges where P(spurious) > threshold until convergence.

    Returns (all_removed, n_iters)
    """
    current_edges = list(init_edges)
    all_removed   = []

    for _ in range(cfg['max_iterations']):
        if not current_edges:
            break

        # Score every remaining existing edge
        proposals = []
        for a, b in current_edges:
            h = get_hidden(tp_fn(goal, a, b), cfg['probe_layer'])
            if probe2_p_spurious(h, tp_name) > threshold:
                proposals.append((a, b))

        current_edges, removed = apply_removals(current_edges, proposals)
        all_removed.extend(removed)

        if not removed:
            break    # converged

    return all_removed, len(all_removed)

print('Iterative helpers ready')


Iterative helpers ready


In [ ]:
from tqdm.auto import tqdm

def gt_coverage(eval_df, label):
    total, reach = 0, 0
    for _, row in eval_df.iterrows():
        edges = set(map(tuple, parse_edges(row.get('dag_edges', ''))))
        gt    = parse_gt(row.get('ground_truth_removed_edges', ''))
        total += len(gt); reach += len(gt & edges)
    print(f'{label}: {reach}/{total} GT reachable ({reach/total*100:.1f}%)' if total else f'{label}: no GT edges')

print('=== GT coverage ===')
gt_coverage(ps_eval, 'ProScript')
gt_coverage(cc_eval, 'CaptainCook')
print()

# ── Main evaluation loop ─────────────────────────────────────────────────────
# For each plan × training prompt variant:
#   - extract (hidden, p_yes) for each existing edge (single forward pass)
#   - cache both probe score and p_yes
#   - sweep thresholds without re-inference

# results[tp_name][dataset][threshold] = list of per-plan metric dicts
results      = {}
# pyes_cache[dataset][goal][(a,b)] = p_yes from tp1_ordering
# (LLM baselines always use tp1_ordering; only extracted once per edge)
pyes_cache   = {'ps': {}, 'cc': {}}

for tp_name, tp_fn in TRAIN_PROMPT_VARIANTS.items():
    results[tp_name] = {'ps': {t: [] for t in cfg['sweep_thresholds']},
                        'cc': {t: [] for t in cfg['sweep_thresholds']}}

    for dataset_tag, eval_df in [('ps', ps_eval), ('cc', cc_eval)]:
        print(f'Running {tp_name} [{dataset_tag}] ...')
        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
            goal       = row['goal']
            steps_list = parse_steps(row)
            init_edges = parse_edges(row.get('dag_edges', ''))
            gt_removed = parse_gt(row.get('ground_truth_removed_edges', ''))

            probe_scores_cache = {}
            for a, b in init_edges:
                h, p_yes = get_hidden_and_pyes(
                    tp_fn(goal, a, b), cfg['probe_layer'])
                probe_scores_cache[(a, b)] = probe2_p_spurious(h, tp_name)
                # Cache p_yes under tp1_ordering for LLM baselines
                if tp_name == 'tp1_ordering':
                    if goal not in pyes_cache[dataset_tag]:
                        pyes_cache[dataset_tag][goal] = {}
                    pyes_cache[dataset_tag][goal][(a, b)] = p_yes

            for thresh in cfg['sweep_thresholds']:
                current, removed_all = list(init_edges), []
                for _ in range(cfg['max_iterations']):
                    if not current: break
                    proposals = [(a, b) for a, b in current
                                 if probe_scores_cache.get((a, b), 0.0) > thresh]
                    current   = [e for e in current if e not in set(proposals)]
                    removed_all.extend(proposals)
                    if not proposals: break
                m = mode2_metrics(set(removed_all), gt_removed, init_edges)
                m['n_iterations'] = sum(1 for _ in range(cfg['max_iterations']))
                results[tp_name][dataset_tag][thresh].append(m)

print('Probe evaluation complete')


=== GT coverage ===
ProScript: 46/46 GT reachable (100.0%)
CaptainCook: 48/48 GT reachable (100.0%)

Running tp1_ordering [ps] ...


  0%|          | 0/91 [00:00<?, ?it/s]

Running tp1_ordering [cc] ...


  0%|          | 0/21 [00:00<?, ?it/s]

Running tp2_parallel [ps] ...


...

Running tp4_flexibility [ps] ...


  0%|          | 0/91 [00:00<?, ?it/s]

Running tp4_flexibility [cc] ...


  0%|          | 0/21 [00:00<?, ?it/s]

Probe evaluation complete


In [ ]:
# ── LLM baselines ────────────────────────────────────────────────────────────
# Both use p_yes from tp1_ordering (cached in pyes_cache during probe eval).
# No additional model inference needed.

llm_results = {
    'direct':   {'ps': {t: [] for t in cfg['sweep_thresholds']},
                 'cc': {t: [] for t in cfg['sweep_thresholds']}},
    'inverted': {'ps': {t: [] for t in cfg['sweep_thresholds']},
                 'cc': {t: [] for t in cfg['sweep_thresholds']}},
}

for dataset_tag, eval_df in [('ps', ps_eval), ('cc', cc_eval)]:
    for _, row in eval_df.iterrows():
        goal       = row['goal']
        init_edges = parse_edges(row.get('dag_edges', ''))
        gt_removed = parse_gt(row.get('ground_truth_removed_edges', ''))
        pyes       = pyes_cache[dataset_tag].get(goal, {})

        for thresh in cfg['sweep_thresholds']:
            # Direct: p_yes < t → spurious (low ordering confidence)
            current, removed_all = list(init_edges), []
            for _ in range(cfg['max_iterations']):
                if not current: break
                proposals = [(a, b) for a, b in current
                             if pyes.get((a, b), 0.5) < thresh]
                current   = [e for e in current if e not in set(proposals)]
                removed_all.extend(proposals)
                if not proposals: break
            llm_results['direct'][dataset_tag][thresh].append(
                mode2_metrics(set(removed_all), gt_removed, init_edges))

            # Inverted: p_yes > t → spurious (yes-bias: high p_yes ≈ spurious edge)
            current, removed_all = list(init_edges), []
            for _ in range(cfg['max_iterations']):
                if not current: break
                proposals = [(a, b) for a, b in current
                             if pyes.get((a, b), 0.5) > thresh]
                current   = [e for e in current if e not in set(proposals)]
                removed_all.extend(proposals)
                if not proposals: break
            llm_results['inverted'][dataset_tag][thresh].append(
                mode2_metrics(set(removed_all), gt_removed, init_edges))

print('LLM baselines complete')
for ds in ('ps', 'cc'):
    best_d = max(cfg['sweep_thresholds'],
                 key=lambda t: summarise(llm_results['direct'][ds][t])['corpus_f1'])
    best_i = max(cfg['sweep_thresholds'],
                 key=lambda t: summarise(llm_results['inverted'][ds][t])['corpus_f1'])
    summarise(llm_results['direct'][ds][best_d],
              label=f'LLM direct   [{ds}] best t={best_d}')
    summarise(llm_results['inverted'][ds][best_i],
              label=f'LLM inverted [{ds}] best t={best_i}')


LLM baselines complete
  LLM direct   [ps] best t=0.7: macro_f1=0.1235  enrichable_f1=0.2882  corpus_f1=0.2165  prec=0.1419  rec=0.4565  fp_rate=0.3186
  LLM inverted [ps] best t=0.35: macro_f1=0.1621  enrichable_f1=0.3781  corpus_f1=0.1875  prec=0.1037  rec=0.9783  fp_rate=0.9828
  LLM direct   [cc] best t=0.7: macro_f1=0.2819  enrichable_f1=0.4229  corpus_f1=0.3973  prec=0.2959  rec=0.6042  fp_rate=0.2257
  LLM inverted [cc] best t=0.35: macro_f1=0.2297  enrichable_f1=0.3446  corpus_f1=0.2588  prec=0.1486  rec=1.0  fp_rate=1.0


In [ ]:
result_rows = []

# Probe 2 rows
for tp_name in TRAIN_PROMPT_VARIANTS:
    for dataset_tag, ds_label in [('ps','ProScript'),('cc','CaptainCook')]:
        for thresh in cfg['sweep_thresholds']:
            s = summarise(results[tp_name][dataset_tag][thresh])
            result_rows.append({'system':'Probe 2','train_variant':tp_name,
                                 'dataset':ds_label,'threshold':thresh,**s})

# LLM baseline rows
for variant_name in ('direct','inverted'):
    label = f'LLM {variant_name}'
    for dataset_tag, ds_label in [('ps','ProScript'),('cc','CaptainCook')]:
        for thresh in cfg['sweep_thresholds']:
            s = summarise(llm_results[variant_name][dataset_tag][thresh])
            result_rows.append({'system':label,'train_variant':'—',
                                 'dataset':ds_label,'threshold':thresh,**s})

results_df = pd.DataFrame(result_rows)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
cols = ['system','train_variant','dataset','threshold',
        'macro_f1','enrichable_f1','corpus_f1','corpus_prec','corpus_rec',
        'fp_rate','avg_iterations']

print('=== BEST CONFIG PER SYSTEM × DATASET (by corpus_f1) ===')
best = results_df.loc[results_df.groupby(
    ['system','train_variant','dataset'])['corpus_f1'].idxmax()]
print(best[cols].sort_values(['dataset','corpus_f1'],ascending=[True,False])
      .to_string(index=False))

print('\n=== LLM BASELINE vs PROBE 2 BEST (ProScript) ===')
ps_best = results_df[results_df.dataset=='ProScript'].loc[
    results_df[results_df.dataset=='ProScript'].groupby(
        ['system','train_variant'])['corpus_f1'].idxmax()]
print(ps_best[cols].sort_values('corpus_f1',ascending=False).to_string(index=False))

print('\n=== LLM BASELINE vs PROBE 2 BEST (CaptainCook) ===')
cc_best = results_df[results_df.dataset=='CaptainCook'].loc[
    results_df[results_df.dataset=='CaptainCook'].groupby(
        ['system','train_variant'])['corpus_f1'].idxmax()]
print(cc_best[cols].sort_values('corpus_f1',ascending=False).to_string(index=False))


=== BEST CONFIG PER SYSTEM × DATASET (by corpus_f1) ===
      system   train_variant     dataset  threshold  macro_f1  enrichable_f1  corpus_f1  corpus_prec  corpus_rec  fp_rate  avg_iterations
     Probe 2    tp1_ordering CaptainCook       0.60    0.4108         0.6162     0.5490       0.5185      0.5833   0.0850            10.0
     Probe 2 tp4_flexibility CaptainCook       0.40    0.3156         0.4735     0.4600       0.4423      0.4792   0.0963            10.0
     Probe 2    tp2_parallel CaptainCook       0.40    0.3298         0.4947     0.4486       0.4068      0.5000   0.1207            10.0
     Probe 2  tp3_dependency CaptainCook       0.45    0.2643         0.3965     0.4000       0.3684      0.4375   0.1167            10.0
  LLM direct               — CaptainCook       0.70    0.2819         0.4229     0.3973       0.2959      0.6042   0.2257             NaN
LLM inverted               — CaptainCook       0.35    0.2297         0.3446     0.2588       0.1486      1.0000   1

In [ ]:
results_df.to_csv(CONTENT / 'mode2_probe2v2_sweep.csv', index=False)
print('Saved: mode2_probe2v2_sweep.csv')

ps_all = results_df[results_df.dataset=='ProScript']
cc_all = results_df[results_df.dataset=='CaptainCook']

print('\n=== Best configs ===')
for ds, sub in [('ProScript', ps_all), ('CaptainCook', cc_all)]:
    best = sub.loc[sub.corpus_f1.idxmax()]
    print(f'{ds}: {best.system} {best.train_variant} t={best.threshold}  '
          f'corpus_f1={best.corpus_f1:.4f}  '
          f'enrichable_f1={best.enrichable_f1:.4f}  '
          f'prec={best.corpus_prec:.4f}  rec={best.corpus_rec:.4f}')

if cfg['retrain_probes']:
    for tp in TRAIN_PROMPT_VARIANTS:
        print(f'  probe2_{tp}_layer{cfg["probe_layer"]}.pkl  saved')


Saved: mode2_probe2v2_sweep.csv

=== Best configs ===
ProScript: Probe 2 tp4_flexibility t=0.6  corpus_f1=0.4037  enrichable_f1=0.4116  prec=0.3492  rec=0.4783
CaptainCook: Probe 2 tp1_ordering t=0.6  corpus_f1=0.5490  enrichable_f1=0.6162  prec=0.5185  rec=0.5833
  probe2_tp1_ordering_layer17.pkl  saved
  probe2_tp2_parallel_layer17.pkl  saved
  probe2_tp3_dependency_layer17.pkl  saved
  probe2_tp4_flexibility_layer17.pkl  saved
